## Parte 2 - Procesamiento de datos

Este notebook lee los datos crudos almacenados en la capa **bronze** durante la 
Parte 1, les aplica tareas de limpieza y enriquecimiento con Pandas, y guarda los 
resultados en las capas **silver** (datos procesados) y **gold** (datos agregados 
listos para consumo), siguiendo la arquitectura Medallion.

**Transformaciones aplicadas:**
1. Conversión de columnas de texto a tipo fecha (`ath_date`, `atl_date`, `last_updated`).
2. Tratamiento de nulos: relleno de `hashing_algorithm` y descarte de `roi`; el resto 
   se conserva de forma documentada.
3. Creación de una columna booleana (`subio_24h`) derivada de la variación de precio.
4. JOIN entre precios y metadata por el campo `id`.
5. Renombrado de columnas duplicadas tras el cruce.


### Extracción desde bronze:

In [1]:
import pandas as pd
from deltalake import DeltaTable

In [2]:
# Rutas de la capa bronze
ruta_markets = "datalake/bronze/coingecko/coins_markets"
ruta_metadata = "datalake/bronze/coingecko/coins_metadata"

# Se lee cada tabla Delta y se pasa a df de Pandas
df_markets = DeltaTable(ruta_markets).to_pandas()
df_metadata = DeltaTable(ruta_metadata).to_pandas()

In [3]:
print("Markets:", df_markets.shape)
print("Metadata:", df_metadata.shape)

Markets: (100, 28)
Metadata: (3, 11)


**Transformaciones:**

*De string a datetime:*

In [4]:
# Columnas con texto que representan fechas
columnas_fecha = ["ath_date", "atl_date", "last_updated"]

for col in columnas_fecha:
    df_markets[col] = pd.to_datetime(df_markets[col], errors="coerce")

In [5]:
df_markets[columnas_fecha].dtypes

ath_date        datetime64[us, UTC]
atl_date        datetime64[us, UTC]
last_updated    datetime64[us, UTC]
dtype: object

*Transformación de datos nulos:*

In [6]:
print("Nulos en metadata:")
print(df_metadata.isnull().sum())

Nulos en metadata:
id                   0
symbol               0
name                 0
hashing_algorithm    1
genesis_date         1
country_origin       0
market_cap_rank      0
categories           0
description_en       0
homepage             0
whitepaper           0
dtype: int64


In [7]:
print("\nNulos en markets (solo columnas con al menos 1 nulo):")
print(df_markets.isnull().sum()[df_markets.isnull().sum() > 0])


Nulos en markets (solo columnas con al menos 1 nulo):
total_volume     7
max_supply      51
roi             91
dtype: int64


In [8]:
# En hashing_algorithm: el nulo significa "no es una blockchain propia"
df_metadata["hashing_algorithm"] = df_metadata["hashing_algorithm"].fillna("No applied")

In [9]:
# En Roi: 91% de nulos y estructura anidada inconsistente, no aporta valor
df_markets = df_markets.drop(columns=["roi"])

In [12]:
print("hashing_algorithm nulos ahora:", df_metadata["hashing_algorithm"].isnull().sum())
print("¿Roi en markets?:", ("roi" in df_markets.columns))

hashing_algorithm nulos ahora: 0
¿Roi en markets?: False


Aclaración:

*max_supply* (51 nulos). El vacío significa que esa moneda no tiene un tope máximo de emisión definido. Es información real, no un dato perdido. Acá reside el peligro de rellenar: si se pusiera 0, se estaría diciendo "esta moneda tiene un máximo de 0 monedas", que es factualmente falso y rompería cualquier análisis posterior. El nulo, en cambio, se interpreta correctamente como "sin límite". Preservar la distinción entre "cero" y "no existe" es un principio básico de calidad de datos.

*Columna nueva por lógica:*

In [13]:
# Nueva columna booleana: True si la moneda subió en las últimas 24h
df_markets["subio_24h"] = df_markets["price_change_percentage_24h"] > 0

In [14]:
# Se verifica cuántas subieron y cuántas no
print(df_markets["subio_24h"].value_counts())

subio_24h
True     70
False    30
Name: count, dtype: int64


In [16]:
# Revisación de columnas relevantes
df_markets[["id", "price_change_percentage_24h", "subio_24h"]].head()

,id,price_change_percentage_24h,subio_24h
0,bitcoin,2.59870,True
1,ethereum,5.15264,True
2,tether,-0.01764,False
3,binancecoin,1.38466,True
4,usd-coin,-0.01524,False


*JOIN entre precios (markets) y metadata, usando 'id' como key:*

In [ ]:
df_completo = df_markets.merge(
    df_metadata,
    on="id",
    how="inner")

In [20]:
df_completo.head()

,id,symbol_x,name_x,image,current_price,market_cap,market_cap_rank_x,fully_diluted_valuation,total_volume,high_24h,...,symbol_y,name_y,hashing_algorithm,genesis_date,country_origin,market_cap_rank_y,categories,description_en,homepage,whitepaper
0,bitcoin,btc,Bitcoin,https://coin-images.coingecko.com/coins/images...,91928993.0,1843296590338313,1,1843299532090356,6.314795e+13,92405129.00,...,btc,Bitcoin,SHA-256,2009-01-03,,1,"[Smart Contract Platform, Layer 1 (L1), FTX Ho...",Bitcoin is the world's first decentralized cry...,[http://www.bitcoin.org],https://bitcoin.org/bitcoin.pdf
1,ethereum,eth,Ethereum,https://coin-images.coingecko.com/coins/images...,2533031.0,305692207827811,2,305692207827811,1.992291e+13,2557187.00,...,eth,Ethereum,Ethash,2015-07-30,,2,"[Smart Contract Platform, Layer 1 (L1), Ethere...","Ethereum is a global, open-source platform for...",[https://www.ethereum.org/],https://ethereum.org/whitepaper/
2,usd-coin,usdc,USDC,https://coin-images.coingecko.com/coins/images...,1488.7,109254500309421,5,109329509923246,2.047463e+13,1492.16,...,usdc,USDC,No applied,NaN,US,5,"[Stablecoins, USD Stablecoin, Solana Ecosystem...",USDC is a fully collateralized US dollar stabl...,[https://www.circle.com/en/usdc],https://www.circle.com/legal/mica-usdc-whitepaper


In [23]:
# Se chequean columnas resultantes aunque ya se destacan algunas que quedaron con "x" y "y"
df_completo.columns.tolist()

['id',
 'symbol_x',
 'name_x',
 'image',
 'current_price',
 'market_cap',
 'market_cap_rank_x',
 'fully_diluted_valuation',
 'total_volume',
 'high_24h',
 'low_24h',
 'price_change_24h',
 'price_change_percentage_24h',
 'market_cap_change_24h',
 'market_cap_change_percentage_24h',
 'circulating_supply',
 'total_supply',
 'max_supply',
 'ath',
 'ath_change_percentage',
 'ath_date',
 'atl',
 'atl_change_percentage',
 'atl_date',
 'last_updated',
 'fecha_extraccion',
 'hora_extraccion',
 'subio_24h',
 'symbol_y',
 'name_y',
 'hashing_algorithm',
 'genesis_date',
 'country_origin',
 'market_cap_rank_y',
 'categories',
 'description_en',
 'homepage',
 'whitepaper']

*Renombrar columnas:*

In [ ]:
# Se descartan las columnas duplicadas (cuyo ocntenido es igual en ambas tablas)
df_completo = df_completo.drop(columns=["symbol_y", "name_y", "market_cap_rank_y"])

In [25]:
# Renombrar las de sufijo _x para dejarlas limpias
df_completo = df_completo.rename(columns={
    "symbol_x": "symbol",
    "name_x": "name",
    "market_cap_rank_x": "market_cap_rank"})

In [26]:
df_completo.columns.tolist()

['id',
 'symbol',
 'name',
 'image',
 'current_price',
 'market_cap',
 'market_cap_rank',
 'fully_diluted_valuation',
 'total_volume',
 'high_24h',
 'low_24h',
 'price_change_24h',
 'price_change_percentage_24h',
 'market_cap_change_24h',
 'market_cap_change_percentage_24h',
 'circulating_supply',
 'total_supply',
 'max_supply',
 'ath',
 'ath_change_percentage',
 'ath_date',
 'atl',
 'atl_change_percentage',
 'atl_date',
 'last_updated',
 'fecha_extraccion',
 'hora_extraccion',
 'subio_24h',
 'hashing_algorithm',
 'genesis_date',
 'country_origin',
 'categories',
 'description_en',
 'homepage',
 'whitepaper']

### Carga a silver:

In [27]:
from deltalake import write_deltalake

In [28]:
ruta_silver_markets = "datalake/silver/coingecko/coins_markets_procesado"

In [29]:
write_deltalake(
    ruta_silver_markets,
    df_markets,
    mode="overwrite")

In [30]:
ruta_silver_completo = "datalake/silver/coingecko/coins_enriquecido"

In [31]:
write_deltalake(
    ruta_silver_completo,
    df_completo,
    mode="overwrite")

In [32]:
print("Markets procesado:", DeltaTable(ruta_silver_markets).to_pandas().shape)

Markets procesado: (100, 28)


In [33]:
print("Enriquecido:", DeltaTable(ruta_silver_completo).to_pandas().shape)

Enriquecido: (3, 35)


*Datos agregados para pasar a gold:*

In [37]:
df_gold_top10 = (
    df_markets
    .sort_values("market_cap", ascending=False)
    .head(10)
    [["market_cap_rank", "id", "name", "current_price", "market_cap", "subio_24h"]]
    .reset_index(drop=True))

In [38]:
df_gold_top10

,market_cap_rank,id,name,current_price,market_cap,subio_24h
0,1,bitcoin,Bitcoin,91928993.00,1843296590338313,True
1,2,ethereum,Ethereum,2533031.00,305692207827811,True
2,3,tether,Tether,1487.58,274146650408721,False
3,4,binancecoin,BNB,834711.00,112504843161968,True
4,5,usd-coin,USDC,1488.70,109254500309421,False
5,6,ripple,XRP,1621.56,100928058640459,True
6,7,solana,Solana,120457.00,69987354466217,True
7,8,tron,TRON,472.98,44862592912430,True
8,9,figure-heloc,Figure Heloc,1540.14,30038421615548,True
9,10,hyperliquid,Hyperliquid,99343.00,22126229802842,True


### Carga a gold:

In [39]:
ruta_gold_top10 = "datalake/gold/coingecko/top10_market_cap"

write_deltalake(
    ruta_gold_top10,
    df_gold_top10,
    mode="overwrite")

In [40]:
DeltaTable(ruta_gold_top10).to_pandas()

,market_cap_rank,id,name,current_price,market_cap,subio_24h
0,1,bitcoin,Bitcoin,91928993.00,1843296590338313,True
1,2,ethereum,Ethereum,2533031.00,305692207827811,True
2,3,tether,Tether,1487.58,274146650408721,False
3,4,binancecoin,BNB,834711.00,112504843161968,True
4,5,usd-coin,USDC,1488.70,109254500309421,False
5,6,ripple,XRP,1621.56,100928058640459,True
6,7,solana,Solana,120457.00,69987354466217,True
7,8,tron,TRON,472.98,44862592912430,True
8,9,figure-heloc,Figure Heloc,1540.14,30038421615548,True
9,10,hyperliquid,Hyperliquid,99343.00,22126229802842,True
